# Interoperability with Qiskit and OpenQASM

Polypus does not require abandoning already-familiar tools: it accepts Qiskit circuits directly, and it can export and import OpenQASM, the standard text format for quantum circuits, in both version 2.0 and version 3.

## Running a Qiskit circuit directly

`run_quantum_circuit` accepts a Qiskit `QuantumCircuit` as is, with no prior conversion:

In [ ]:
import polypus
from qiskit import QuantumCircuit

qc_qiskit = QuantumCircuit(2)
qc_qiskit.h(0)
qc_qiskit.cx(0, 1)
qc_qiskit.measure_all()

result = polypus.run_quantum_circuit(qc_qiskit, shots=1000, infrastructure="local")
print(result.counts[0])

The result is the same Bell state from [`01_first_circuit.ipynb`](01_first_circuit.ipynb), only `'00'` and `'11'`, this time built with Qiskit's API instead of with `polypus.Circuit`.

## Exporting to OpenQASM

Any Polypus `Circuit` can be exported to OpenQASM 2.0, the same text format Qiskit uses:

In [ ]:
bell = polypus.Circuit(2).h(0).cx(0, 1).measure_all()
print(bell.to_qasm2())

The result is plain text, readable and portable to any tool that understands OpenQASM 2.0.

## Importing from OpenQASM

`Circuit.from_qasm2` does the reverse, and it also guarantees an exact round trip: exporting, importing, and exporting again produces the same text, byte for byte.

In [ ]:
qasm_text = bell.to_qasm2()
reimported_bell = polypus.Circuit.from_qasm2(qasm_text)

print(reimported_bell.to_qasm2() == qasm_text)

An imported circuit is a regular `Circuit`: gates can keep being chained onto it like any other.

## OpenQASM 3

Polypus also exports and imports OpenQASM 3, the newer version of the format, with `to_qasm3` and `Circuit.from_qasm3`. The round trip is exact here too: exporting, importing, and exporting again produces the same text, byte for byte.

In [ ]:
qasm3_text = bell.to_qasm3()
print(qasm3_text)

bell_from_qasm3 = polypus.Circuit.from_qasm3(qasm3_text)
print(bell_from_qasm3.to_qasm3() == qasm3_text)

The main difference with OpenQASM 2.0 is that version 3 can also store free parameters, angles left open to be given a value later, which [`06a_variational_training.ipynb`](06a_variational_training.ipynb) introduces. Polypus supports the part of OpenQASM 3 that describes a circuit as a straight sequence of gates with the measurements at the end: a program with conditionals or loops, for example, is rejected with a `ValueError` that names the construct and its line.

## Bringing a Qiskit circuit in through QASM

`from_qasm2` also accepts the text Qiskit produces with `qasm2.dumps`, not just Polypus's own. This is useful for converting a Qiskit circuit into a native `Circuit`, for example to keep building it with the method chaining seen in [`00a`](00a_basic_programming.ipynb):

In [ ]:
from qiskit import qasm2

qiskit_qasm = qasm2.dumps(qc_qiskit)
native_circuit = polypus.Circuit.from_qasm2(qiskit_qasm)

print(native_circuit.to_qasm2())

On import, Polypus keeps every gate under its own name, `p`, `u1`, `u2`, `u`, and `id` included: only QASM's own `U`/`CX` builtins get normalized to `u`/`cx`. The result is still the same circuit.

Besides OpenQASM, Polypus can export to QIR, a lower-level format aimed at compilers. It is outside the scope of this series; the [circuits guide](https://github.com/Bahia-Software/polypus/blob/main/docs/guide/circuits.md#export) explains it.

## Drawing with Qiskit

[`02a`](02a_gates_and_circuits.ipynb) showed that a Polypus circuit draws itself. Readers who prefer Qiskit's matplotlib drawings can still go through Qiskit: export to QASM, import back with `qasm2.loads`, the inverse of `qasm2.dumps` used above, and call `draw("mpl")`, which needs `matplotlib` and `pylatexenc` installed. Qiskit's parser does not recognize some of the gates Polypus exports, like `swap`, `cp`, `rzz`, or `rxx`, unless `custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS` is passed:

```python
qiskit_circuit = qasm2.loads(
    bell.to_qasm2(), custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS
)
qiskit_circuit.draw("mpl")
```

## Summary

This notebook showed several interoperability paths: running a Qiskit circuit directly, exporting a Polypus circuit to OpenQASM 2.0 and OpenQASM 3, importing both Polypus's own OpenQASM and the one Qiskit generates, and drawing a Polypus circuit with matplotlib through Qiskit, as an alternative to the native drawing. Readers who already know Qiskit and want the full mapping, gates, execution, and training included, have the reference guide [`from_qiskit_to_polypus.ipynb`](from_qiskit_to_polypus.ipynb).

The next section covers where these circuits run: locally, or spread across several simulators or QPUs.

## Next step

Continue with: [`03a_local_execution.ipynb`](03a_local_execution.ipynb).